In [7]:
import numpy as np
import pandas as pd
from pathlib import Path
import os
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from matplotlib import pyplot as plt

In [8]:
#csv_dir = Path(__file__).resolve().parent/'data'/'ds_salaries.csv'
df = pd.read_csv('ds_salaries.csv')
df = df[['work_year', 'experience_level', 'employment_type', 'remote_ratio', 'company_size', 'job_title', 'salary_in_usd']]
        
print(df['work_year'].unique())
print('_________________________')
print(df['experience_level'].unique())
print('_________________________')
print(df['employment_type'].unique())
print('_________________________')
print(df['remote_ratio'].unique())
print('_________________________')
print(df['company_size'].unique())
print('_________________________')
print(df['job_title'].value_counts().head(10))

[2020 2021 2022]
_________________________
<StringArray>
['MI', 'SE', 'EN', 'EX']
Length: 4, dtype: str
_________________________
<StringArray>
['FT', 'CT', 'PT', 'FL']
Length: 4, dtype: str
_________________________
[  0  50 100]
_________________________
<StringArray>
['L', 'S', 'M']
Length: 3, dtype: str
_________________________
job_title
Data Scientist                143
Data Engineer                 132
Data Analyst                   97
Machine Learning Engineer      41
Research Scientist             16
Data Science Manager           12
Data Architect                 11
Machine Learning Scientist      8
Big Data Engineer               8
Data Science Consultant         7
Name: count, dtype: int64


In [9]:
top_jobs = df['job_title'].value_counts().nlargest(7).index.tolist()
df['job_group'] = df['job_title'].apply(lambda x: x if x in top_jobs else 'other')
df = df.drop(['job_title'],axis=1)
df.head()

,work_year,experience_level,employment_type,remote_ratio,company_size,salary_in_usd,job_group
0,2020,MI,FT,0,L,79833,Data Scientist
1,2020,SE,FT,0,S,260000,other
2,2020,SE,FT,50,M,109024,other
3,2020,MI,FT,0,S,20000,other
4,2020,SE,FT,50,L,150000,Machine Learning Engineer


In [10]:
year_map = {2020: 1, 2021: 2, 2022: 3}
df['ord_year'] = df['work_year'].map(year_map)

exp_map = {"EN": 1, "MI": 2, "SE": 3, "EX": 4}
df["experience_rank"] = df["experience_level"].map(exp_map)
df = df.drop(['experience_level', 'work_year'], axis=1)
df.head()

,employment_type,remote_ratio,company_size,salary_in_usd,job_group,ord_year,experience_rank
0,FT,0,L,79833,Data Scientist,1,2
1,FT,0,S,260000,other,1,3
2,FT,50,M,109024,other,1,3
3,FT,0,S,20000,other,1,2
4,FT,50,L,150000,Machine Learning Engineer,1,3


In [11]:
df = pd.get_dummies(df, columns=['job_group', 'company_size', 'remote_ratio', 'employment_type'])
df.head()

,salary_in_usd,ord_year,experience_rank,job_group_Data Analyst,job_group_Data Architect,job_group_Data Engineer,job_group_Data Science Manager,job_group_Data Scientist,job_group_Machine Learning Engineer,job_group_Research Scientist,...,company_size_L,company_size_M,company_size_S,remote_ratio_0,remote_ratio_50,remote_ratio_100,employment_type_CT,employment_type_FL,employment_type_FT,employment_type_PT
0,79833,1,2,False,False,False,False,True,False,False,...,True,False,False,True,False,False,False,False,True,False
1,260000,1,3,False,False,False,False,False,False,False,...,False,False,True,True,False,False,False,False,True,False
2,109024,1,3,False,False,False,False,False,False,False,...,False,True,False,False,True,False,False,False,True,False
3,20000,1,2,False,False,False,False,False,False,False,...,False,False,True,True,False,False,False,False,True,False
4,150000,1,3,False,False,False,False,False,True,False,...,True,False,False,False,True,False,False,False,True,False


In [12]:
df['salary'] = np.log1p(df['salary_in_usd'])
df = df.drop(['salary_in_usd'], axis=1)
df.head()

,ord_year,experience_rank,job_group_Data Analyst,job_group_Data Architect,job_group_Data Engineer,job_group_Data Science Manager,job_group_Data Scientist,job_group_Machine Learning Engineer,job_group_Research Scientist,job_group_other,...,company_size_M,company_size_S,remote_ratio_0,remote_ratio_50,remote_ratio_100,employment_type_CT,employment_type_FL,employment_type_FT,employment_type_PT,salary
0,1,2,False,False,False,False,True,False,False,False,...,False,False,True,False,False,False,False,True,False,11.287705
1,1,3,False,False,False,False,False,False,False,True,...,False,True,True,False,False,False,False,True,False,12.468441
2,1,3,False,False,False,False,False,False,False,True,...,True,False,False,True,False,False,False,True,False,11.599332
3,1,2,False,False,False,False,False,False,False,True,...,False,True,True,False,False,False,False,True,False,9.903538
4,1,3,False,False,False,False,False,True,False,False,...,False,False,False,True,False,False,False,True,False,11.918397


In [13]:
def preprocessing():
    predict = 'salary'
    X = np.array(df.drop([predict], axis=1))
    Y = np.array(df[predict])
    return train_test_split(X, Y, test_size=0.2, random_state=42)

In [14]:
df['experience_rank'].nunique

<bound method IndexOpsMixin.nunique of 0      2
1      3
2      3
3      2
4      3
      ..
602    3
603    3
604    3
605    3
606    2
Name: experience_rank, Length: 607, dtype: int64>

In [15]:

x_train, x_test, y_train, y_test = preprocessing()
model = LinearRegression()
model.fit(x_train, y_train)
score = model.score(x_test, y_test)
score


0.32219317800418246

In [16]:
predictions = model.predict(x_test)
for i in range(len(predictions)):
    print(f'{predictions[i]}:{y_test[i]}')

11.784494763318293:11.8511889534843
11.784494763318293:11.813037464800539
11.27151983740028:11.51293546492023
11.618454441679722:12.506180941677357
11.265457170125392:10.166082559611265
11.183818166068173:11.938199736300925
10.439441955670283:10.845874789131557
10.62227026417697:10.812693244687052
11.37255889031589:11.250794173362452
10.417527211290029:10.987036963527192
10.699615626945409:11.046547255939533
11.784494763318293:11.612779849672751
11.784494763318293:11.827699707793888
11.182383247536258:10.119566206128216
11.547068177239492:11.906736257934913
11.784494763318293:12.07254696717504
10.936929327166316:10.990432036336555
11.337784172893416:11.626263078808801
10.925848299891015:10.323151380044106
11.819843243565874:11.60167782556802
11.322978972744192:11.599332492710968
10.933182581727813:9.392745258631441
11.784494763318293:11.944714374881176
11.359908157415388:11.982935344196433
11.000989857561484:9.798182590890704
10.84922765583421:11.256289760923773
11.712167760550269:11.7

In [17]:
predictions =  model.predict(x_test)
x = 0
y = 0
for i in range(len(predictions)):
    x += predictions[i]
    y += y_test[i]
x = x / len(predictions)
y = y / len(predictions)
print(f'{x}_{y}')

11.372872912845907_11.317229761042409


In [18]:
y_test.shape

(122,)

In [19]:
predictions =  model.predict(x_test)
predictions

array([11.78449476, 11.78449476, 11.27151984, 11.61845444, 11.26545717,
       11.18381817, 10.43944196, 10.62227026, 11.37255889, 10.41752721,
       10.69961563, 11.78449476, 11.78449476, 11.18238325, 11.54706818,
       11.78449476, 10.93692933, 11.33778417, 10.9258483 , 11.81984324,
       11.32297897, 10.93318258, 11.78449476, 11.35990816, 11.00098986,
       10.84922766, 11.71216776, 11.13581895, 11.60187569, 11.37255889,
       10.84733156, 11.60187569, 11.81984324, 10.9959715 , 10.95945164,
       11.60187569, 11.39468287, 11.60187569, 10.66138099, 11.73429175,
       11.80661875, 11.80661875, 11.42459179, 11.65498007, 11.60187569,
       11.16441812, 11.15120912, 11.33778417, 11.36892017, 11.90823156,
       11.60187569, 10.98776536, 11.30023189, 11.60187569, 11.78449476,
       11.60187569, 11.68345571, 12.1246774 , 10.77403247, 11.4927063 ,
       11.50952019, 10.66138099, 10.46427872, 11.6354565 , 11.78506853,
       11.30023189, 11.59431912, 11.11761282, 11.60187569, 11.14

formula for simple linear regression == y = ax+b
formula for multiple linear regression == y = a1x + a2p + a3k + .... + b
a = weight, b=bias, intercept(coefficient(coef))

In [20]:
coef = model.coef_
bias = model.intercept_
pred_test = (coef * x_test[0] + bias)/20
model_pred = model.predict([x_test[0]])

column = 0
for i in range(len(coef)):
    column += coef[i] * x_test[0][i] + bias

print(f"{column/20} : {model_pred}")


10.076240772413644 : [11.784494763318293]
